# SIENA Dataset loading
The test and validation data of this work was found in the *PhysioNet*, precisely at the following link: [https://physionet.org/content/siena-scalp-eeg/1.0.0/PN00/PN00-1.edf](https://physionet.org/content/siena-scalp-eeg/1.0.0/PN00/PN00-1.edf). My main objective is to load the *dataset* and structure it the same way teached on classroom.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from pyedflib import highlevel

# Suporta execução tanto a partir de src/jupyter/ quanto da raiz do repositório
DATA_PATH = Path("../../data/PN00-2.edf") if Path("../../data/PN00-2.edf").exists() else Path("./data/PN00-2.edf")

signals, signal_headers, header = highlevel.read_edf(str(DATA_PATH))
print(f"Arquivo carregado: {DATA_PATH}")
print(f"Total de canais: {len(signals)}")
print(f"Amostras por canal: {len(signals[0])}")



---


In [ ]:
# Metadados e identificação de cada coluna (canal) em formato tabular
df_channels = pd.DataFrame([
    {
        "#": idx + 1,
        "Canal": h["label"],
        "Unidade": h["dimension"],
        "Freq (Hz)": h["sample_frequency"],
        "Faixa Mín": h["physical_min"],
        "Faixa Máx": h["physical_max"],
    }
    for idx, h in enumerate(signal_headers)
])

df_channels


---

In [ ]:
# Construção do DataFrame dos dados (amostras como linhas, canais como colunas)
channel_names = [h["label"] for h in signal_headers]
df_signals = pd.DataFrame(dict(zip(channel_names, signals)))
df_signals.index.name = "Amostra"

# Exibição das primeiras 10 linhas do dataset
df_signals.head(10)


---
## 2. Cálculo e Estruturação das Epochs (Trials)
Para comparar as séries temporais do EEG real com os dados simulados pelo Autômato Celular (*Cellular Automata*), fatiamos o sinal contínuo em *epochs*.

Apenas os **29 canais de EEG** (descartando EKG, SPO2, HR e canais de teste).

In [ ]:
# 1. Filtrar apenas canais de EEG
eeg_indices = [i for i, h in enumerate(signal_headers) if h["label"].startswith("EEG")]
eeg_labels = [signal_headers[i]["label"] for i in eeg_indices]
eeg_signals = np.array([signals[i] for i in eeg_indices], dtype=np.float32)

# 2. Configurações da Janela Temporal
FS = 512.0          # Taxa de amostragem em Hz
WINDOW_SEC = 2.0    # Duração de cada trial em segundos
OVERLAP = 0.0       # Percentual de sobreposição (0.0 = sem overlap, 0.5 = 50%)

samples_per_epoch = int(WINDOW_SEC * FS)
step = int(samples_per_epoch * (1.0 - OVERLAP))
total_samples = eeg_signals.shape[1]
n_epochs = (total_samples - samples_per_epoch) // step + 1

# 3. Montar a matriz 3D de epochs: (n_epochs, n_canais, n_amostras)
epochs = np.empty((n_epochs, len(eeg_labels), samples_per_epoch), dtype=np.float32)
for idx in range(n_epochs):
    start = idx * step
    epochs[idx] = eeg_signals[:, start : start + samples_per_epoch]

print("=== ESTRUTURA DAS EPOCHS ===")
print(f"Canais selecionados: {len(eeg_labels)} canais de EEG")
print(f"Duração por epoch:   {WINDOW_SEC} s ({samples_per_epoch} amostras)")
print(f"Total de epochs:    {n_epochs}")
print(f"Shape final:        {epochs.shape} -> (trials, canais, amostras)")
print(f"Tipo dos dados:     {epochs.dtype}")


In [ ]:
# Caminho de saída (pasta output/)
OUTPUT_DIR = Path("../../output") if Path("../../output").exists() else Path("./output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

npy_path = OUTPUT_DIR / "eeg_epochs.npy"
np.save(npy_path, epochs)
print(f"Array bruto salvo em: {npy_path.resolve()}")
